[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilay47/marginally-realistic/blob/main/cvpr27/notebooks/nb0_benchmark.ipynb)

# nb0: benchmark, floors, self-test, video-metric decision
**Use a GPU runtime** (the I3D self-check and the real-reference features are computed here and cached for every other notebook). Builds the data cache (10,000 train + 2,048 held-out clips), decides FVD vs KID **once**, then computes E1: the floors, the corruption self-test and the pixel-noise sweep on the held-out real clips. Every number is computed in this run.

## 1. Runtime

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no GPU (CPU run: the video-metric check will be very slow)")
print("cuda available:", torch.cuda.is_available())

## 2. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/physgen_cvpr27"
os.makedirs(ROOT, exist_ok=True)

## 3. Code

In [ ]:
import os
if os.path.exists("/content/repo"):
    !git -C /content/repo pull
else:
    !git clone https://github.com/nilay47/marginally-realistic.git /content/repo
%cd /content/repo
!git log -1 --format="commit %H%n%s"

## 4. Install

In [ ]:
# torch / torchvision come with Colab. pytorchvideo is the second I3D source for FVD (see videofeat.py).
!pip install -q scipy matplotlib pytorchvideo

## 5. Run
Order: cache -> video-metric decision (tries the StyleGAN-V I3D torchscript, then pytorchvideo I3D-R50, then KID; each must pass a behavioural self-check) -> floors / corruptions / noise sweep. Stops (Rule 5) if the extractor's invalid-frame fraction on real data exceeds 1% or a clean floor exceeds 1e-2.

In [ ]:
!cd /content/repo && python -u -m cvpr27.benchmark

## 6. Summary

In [ ]:
import json
b = json.load(open(f"{ROOT}/benchmark/floors.json"))
vm = b["video_metric"]
print("VIDEO METRIC:", vm["kind"].upper(), "| flavor:", vm["flavor"], "| fallback_to_kid:", vm["fallback_to_kid"])
for a in vm["attempts"]:
    print("  attempt", a["flavor"], "->", "OK" if a.get("ok") else "FAILED", a.get("reason") or {k: v for k, v in a["check"].items() if k != "ok"})
print("\ngit:", b["git"]["commit"][:8], "| rule-5 gate failures:", b["rule5_gate_failures"])
print(open(f"{ROOT}/benchmark/selftest_table.md").read())